In [42]:
import nltk
import string
import pickle
import pandas as pd
import os

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

CSV_PATH ="./financial_dataset.csv"
MODEL_PATH ="./new_model.pkl"

In [43]:
df = pd.read_csv(CSV_PATH)
df.head()

,Statement,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,$SPY wouldn't be surprised to see a green close,positive
4,Shell's $70 Billion BG Deal Meets Shareholder ...,negative


In [44]:
# preprocessing
lemmatizer = WordNetLemmatizer()
stemmer = SnowballStemmer('english')

def get_label(tag):
    if tag.startswith('j'):
        return 'a'
    elif tag.startswith('r') or tag.startswith('v') or tag.startswith('n'):
        return tag[0]
    else:
        return 'n'

def lemmatizing(word_list):
    lemma_list = []
    tagged = pos_tag(word_list)

    for word, tag in tagged:
        label = get_label(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemma_list.append(result)
    return lemma_list

    
def preprocessing(sentence):
    eng_stopwords = stopwords.words('english')
    punctuation = string.punctuation

    word_list = word_tokenize(sentence.lower())
    removed_stopwords = [token for token in word_list if token not in eng_stopwords]
    removed_punctuation = [token for token in removed_stopwords if token not in punctuation]
    removed_numbers = [token for token in removed_punctuation if token.isalpha()]

    # stemmer
    stemming = [stemmer.stem(token) for token in removed_numbers]

    lemma_list = lemmatizing(removed_numbers)

    return lemma_list

In [45]:
sentence = "hello bro are you good?"
result = preprocessing(sentence)

print(result)

['hello', 'bro', 'good']


In [46]:
x = df["Statement"]
y = df["Sentiment"]

all_sentence = " ".join(x)
all_token = preprocessing(all_sentence)
freq_dist = FreqDist(all_token)

print(freq_dist.most_common(10))

[('eur', 747), ('mn', 459), ('profit', 368), ('sale', 338), ('company', 321), ('net', 297), ('say', 296), ('finnish', 266), ('million', 243), ('year', 234)]


In [47]:
def extract_feature(statement):
    features = {}
    for word in freq_dist.keys():
        features [word] = (word in statement)
    return features

In [48]:
# buat fitur set
feature_sets = []
for (statement, sentiment) in zip(x,y):
    features = extract_feature(preprocessing(statement))
    feature_sets.append((features, sentiment))
print(feature_sets[0])

({'geosolutions': True, 'technology': True, 'leverage': True, 'benefon': True, 'gps': True, 'solution': True, 'provide': True, 'location': True, 'base': True, 'search': True, 'community': True, 'platform': True, 'relevant': True, 'multimedia': True, 'content': True, 'new': True, 'powerful': True, 'commercial': True, 'model': True, 'esi': False, 'low': False, 'bk': False, 'real': False, 'possibility': False, 'last': False, 'quarter': False, 'componenta': False, 'net': False, 'sale': False, 'double': False, 'period': False, 'year': False, 'earlier': False, 'move': False, 'zero': False, 'profit': False, 'loss': False, 'spy': False, 'would': False, 'surprise': False, 'see': False, 'green': False, 'close': False, 'shell': False, 'billion': False, 'bg': False, 'deal': False, 'meet': False, 'shareholder': False, 'skepticism': False, 'ssh': False, 'communication': False, 'security': False, 'corp': False, 'stock': False, 'exchange': False, 'release': False, 'october': False, 'pm': False, 'compa

In [49]:
# split 80:20
train_count = int(len(feature_sets) * 0.8)
train_set = feature_sets[:train_count]
test_set = feature_sets[train_count:]

In [50]:
# main function

def write_statement():
    while True:
        statement = input("write your statement")
        if(len(statement.split()) > 2):
            print("invalid input, write at least 2 words")
        else:
            return statement

    
def train_model():
    classifier = NaiveBayesClassifier.train(train_set)
    test_accuracy = accuracy(classifier, test_set)

    classifier.show_most_informative_features(7)
    print(f"Accuracy: {test_accuracy * 100}%")

    with open(MODEL_PATH, "wb") as f:
        pickle.dump(classifier, f)
    return classifier

def show_pos_tag(statement):
    print("\nPOS TAG:")

    tokens = [t for t in word_tokenize(statement) if t.isalpha()]
    tagged = pos_tag(tokens)

    for word, tag in tagged:
        print(f"- {word} : {tag}")
    return tokens

def show_syn_ant(tokens):
    print(f"Synonym and Antonym:")

    for token in tokens:
        synonym = []
        antonym = []
        synset = wordnet.synsets(token)

        for syn in synset:
            for lemma in syn.lemmas():
                synonym.append(lemma.name())
                for ant in lemma.antonyms():
                    antonym.append(ant.name)
        
        print(f"\nWord : {tokens}")

        if synonym:
            print("Synonym: ")
            print(synonym[:5])
        else: 
            print("no antonyms")

        if antonym:
            print("Antonym: ")
            print(antonym[:5])
        else: 
            print("no antonyms")

def analyze_statement(statement, classifier):
    if len(statement) == 0:
        print("No statement")
        return
    

    tokens = show_pos_tag(statement)
    show_syn_ant(tokens)

    preprocessed_text = preprocessing(statement)
    extracted = extract_feature(preprocessed_text)
    prediction = classifier.classify(extracted)

    print(f"\nPredicted Sentiment: {prediction}")

In [51]:
# menu

statement = ""
classifier = None

while True:
    print("\n1. write your statement")
    print("2. analyze your statement")
    print("3. exit")

    choice = input (">> ")

    if(choice == "1"):
        statement = write_statement()
        print(f"Current Statement: {statement}")

        if os.path.exists(MODEL_PATH):
            with open(MODEL_PATH, "rb") as f:
                classifier = pickle.load(f)
        else:
            classifier = train_model()

    elif(choice == "2"):
        analyze_statement(statement, classifier)

    elif(choice == "3"):
        print("program ended..")
        break

    else:
        print("Invalid input, press 1-3")


1. write your statement
2. analyze your statement
3. exit
Current Statement: hello guys
Most Informative Features
                decrease = True           negati : positi =     31.8 : 1.0
                    fell = True           negati : positi =     30.4 : 1.0
                     lay = True           negati : positi =     19.8 : 1.0
                   staff = True           negati : positi =     18.4 : 1.0
                  recall = True           negati : positi =     16.3 : 1.0
                   lower = True           negati : positi =     13.7 : 1.0
                    drop = True           negati : positi =     13.5 : 1.0
Accuracy: 77.90055248618785%

1. write your statement
2. analyze your statement
3. exit

POS TAG:
- hello : NN
- guys : NNS
Synonym and Antonym:

Word : ['hello', 'guys']
Synonym: 
['hello', 'hullo', 'hi', 'howdy', 'how-do-you-do']
no antonyms

Word : ['hello', 'guys']
Synonym: 
['guy', 'cat', 'hombre', 'bozo', 'Guy']
no antonyms

Predicted Sentiment: positi